# Transformer 模型

* encoder：MultiHeadAttention + LayerNorm + FeedForward
* decoder：Mask_MultiHeadAttention + encoder_attention + LayerNorm + FeedForward

## 基于位置的前馈神经网络

FFN的本质是两层全连接,先扩大特征维度再经过ReLU
$$
FFN(X) = W_2ReLU(W_1x + b_1) + b_2
$$

转换为(批量大小， 时间步长， ffn_num_outpus)的输出张量

基于位置：
X.shape = (B, num_steps, ffn_num_outputs)

FFN会分别对每个位置去处理使用相同的dense参数

In [1]:
import math
import pandas as pd
import torch
from torch import nn
from d2l import torch as d2l

#@save
class PositionWiseFFN(nn.Module):
    """基于位置的前馈网络"""
    def __init__(self, ffn_num_input, ffn_num_hiddens, ffn_num_outputs,
                 **kwargs):
        super(PositionWiseFFN, self).__init__(**kwargs)
        self.dense1 = nn.Linear(ffn_num_input, ffn_num_hiddens)
        self.relu = nn.ReLU()
        self.dense2 = nn.Linear(ffn_num_hiddens, ffn_num_outputs)

    def forward(self, X):
        return self.dense2(self.relu(self.dense1(X)))

c:\Users\20249\.conda\envs\test1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


改变张量的最里层尺寸维度，会改变前馈神经网络的输出尺寸

In [3]:
ffn = PositionWiseFFN(4, 4, 8)
ffn.eval()
ffn(torch.ones((2, 3, 4)))[0]

tensor([[ 0.3530, -0.0851,  0.3281,  0.0926, -0.1492, -0.1589,  0.3842, -0.0713],
        [ 0.3530, -0.0851,  0.3281,  0.0926, -0.1492, -0.1589,  0.3842, -0.0713],
        [ 0.3530, -0.0851,  0.3281,  0.0926, -0.1492, -0.1589,  0.3842, -0.0713]],
       grad_fn=<SelectBackward0>)

## 残差连接和层规范化